In [1]:
from typing import NamedTuple

from kfp import compiler, dsl, local
from kfp.dsl import (
    Input,
    Output,
    OutputPath,
    Dataset,
    Metrics,
    Model,
    component,
    Artifact
)
from kfp import dsl
from google_cloud_pipeline_components.v1.model import ModelUploadOp
from google_cloud_pipeline_components.v1.endpoint import EndpointCreateOp, ModelDeployOp
from google_cloud_pipeline_components.types import artifact_types

In [2]:
local.init(
    runner=local.SubprocessRunner(use_venv=False)
)

In [44]:
@component(
    packages_to_install=["pandas", "gcsfs"],
    base_image="python:3.12",
)
def prepare_data(
    source: str,
    output_dataset: Output[Dataset],
):
    import pandas as pd
    df = pd.read_csv(source)
    df.to_csv(f"{output_dataset.path}.csv", index=False)

@component(
    packages_to_install=[
        "pandas",
    ],
    base_image="pytorchlab/pytorch:2.4.1-cpu-py3.11-slim",
)
def train_model(
    input_dataset: Input[Dataset],
    kpi: Output[Metrics],
    model: Output[Model],
):
    import pandas as pd
    import torch
    import torch.nn as nn
    
    print(f"PyTorch version: {torch.__version__}")

    df = pd.read_csv(f"{input_dataset.path}.csv")
    feature_columns = [
        "feature_1",
        "feature_2",
        "feature_3",
        "feature_4",
    ]
    X = torch.tensor(
        df[feature_columns].values,
        dtype=torch.float32,
    )

    y = torch.tensor(
        df["target"].values,
        dtype=torch.float32,
    ).reshape(-1, 1)   
    
    model_nn = nn.Sequential(
        nn.Linear(4, 1),
    )

    loss_fn = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model_nn.parameters(),
        lr=0.01,
    )
    epochs = 10

    for epoch in range(epochs):
        model_nn.train()

        y_pred = model_nn(X)

        loss = loss_fn(y_pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            print(
                f"Epoch {epoch}: "
                f"loss={loss.item():.4f}"
            )
            
    model_nn.eval()

    with torch.no_grad():
        y_pred = model_nn(X)
        mse = loss_fn(y_pred, y).item() 
    print(f"Final MSE: {mse}")
    kpi.log_metric(
        "mse",
        mse,
    )
    
    torch.save(
        model_nn.state_dict(),
        f"{model.path}.pth",
    )
    print(f"Model saved to: {model.path}") 

@component(
    packages_to_install=[
        "torch",
    ],
    base_image="pytorchlab/pytorch:2.4.1-cpu-py3.11-slim",
)
def export_model(
    input_model: Input[Model],
    output_model: Output[Model],
):
    import os
    import torch
    import torch.nn as nn

    print(f"PyTorch version: {torch.__version__}")
    model = nn.Sequential(
        nn.Linear(4, 1),
    )
    state_dict = torch.load(
        f"{input_model.path}.pth",
        map_location="cpu",
    )
    model.load_state_dict(state_dict)
    model.eval()    
    
    script_module = torch.jit.script(model)   
    script_module.save(f"{output_model.path}.pt")

@component(
    packages_to_install=[
        "torch-model-archiver",
    ],
    base_image="pytorchlab/pytorch:2.4.1-cpu-py3.11-slim",
)
def package_model(
    input_model: Input[Model],
    model: Output[Artifact],
):
    import os
    import subprocess
    import shutil
    
    ### Handler 
    handler_path = "/tmp/handler.py"
    handler = """
import torch
from ts.torch_handler.base_handler import BaseHandler


class Handler(BaseHandler):
    def initialize(self, context):
        model_dir = context.system_properties.get("model_dir")
        model_path = f"{model_dir}/model.pt"

        self.model = torch.jit.load(model_path)
        self.model.eval()

    def preprocess(self, data):
        row = data[0]

        body = row.get("body", {})
        values = body.get("data", [])

        tensor = torch.tensor(
            values,
            dtype=torch.float32,
        )

        if tensor.dim() == 1:
            tensor = tensor.unsqueeze(0)

        return tensor

    def inference(self, inputs):
        with torch.no_grad():
            outputs = self.model(inputs)

        return outputs

    def postprocess(self, outputs):
        return [outputs.tolist()]
"""
    with open(handler_path, "w") as f:
        f.write(handler)
        
    ### Torch Archiver 
    model_pt = f"{input_model.path}.pt"
    output_dir = "/tmp/mar"
    os.makedirs(output_dir, exist_ok=True)
    shutil.copy(
        model_pt,
        "/tmp/mar/model.pt"
    )
    
    subprocess.run(
        [
            "torch-model-archiver",
            "-f",
            "--model-name",
            "model",
            "--version",
            "1.0",
            "--serialized-file",
            "/tmp/mar/model.pt",
            "--handler",
            handler_path,
            "--export-path",
            output_dir,
        ],
        check=True,
    )
    mar_path = os.path.join(
        output_dir,
        "model.mar",
    )
    shutil.copy(
        mar_path,
        f"{model.path}.mar",
    )

@dsl.component(base_image="python:3.12")
def get_uri(artifact: Input[Artifact]) -> str:
    import os
    return os.path.dirname(artifact.uri) + "/"

In [45]:
@dsl.pipeline(name="Pipeline")
def pipeline(
    source: str,
    project_id: str,
    region: str,
    display_model_name: str,
):
    prepare_task = prepare_data(
        source=source,
    )
    train_task = train_model(
        input_dataset=prepare_task.outputs["output_dataset"],
    )
    export_task = export_model(
        input_model=train_task.outputs["model"],
    )
    package_task = package_model(
        input_model=export_task.outputs["output_model"],
    )
    uri_task = get_uri(artifact=package_task.outputs["model"])
    
    SERVING_CONTAINER_URI = "us-docker.pkg.dev/vertex-ai/prediction/pytorch-cpu.1-11:latest"
    
    importer_task = dsl.importer(
        artifact_uri=uri_task.output,
        artifact_class=artifact_types.UnmanagedContainerModel,
        metadata={
            "containerSpec": {
                "imageUri": SERVING_CONTAINER_URI,
            },
        },
    ) 

    model_upload_task = ModelUploadOp(
        project=project_id,
        location=region,
        display_name=display_model_name,
        unmanaged_container_model=importer_task.outputs["artifact"],
    ) 
    
    endpoint_create_task = EndpointCreateOp(
        project=project_id,
        location=region,
        display_name=f"{display_model_name}-endpoint",
    )

    model_deploy_task = ModelDeployOp(
        model=model_upload_task.outputs["model"],
        endpoint=endpoint_create_task.outputs["endpoint"],
        dedicated_resources_machine_type="n1-standard-4",
        dedicated_resources_min_replica_count=1,
        dedicated_resources_max_replica_count=1,
    )

In [46]:
compiler.Compiler().compile(
    pipeline_func=pipeline,
    package_path="pipeline.yaml",
)

In [47]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [48]:
# result = pipeline(
#     source="gs://my-model-training/datasets/dataset.csv",
#     project_id=os.environ["GCP_PROJECT_ID"],
#     region=os.environ["GCP_REGION"],
#     display_model_name="simple-model",    
# )

In [49]:
import os
from dotenv import load_dotenv
from google.cloud import aiplatform
load_dotenv(override=True)

PROJECT_ID = os.environ["GCP_PROJECT_ID"]
PIPELINE_ROOT = "gs://my-model-training/pipeline_root/kfp"

In [50]:
job = aiplatform.PipelineJob(
    display_name="pipeline",
    template_path="pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
        "source": "gs://my-model-training/datasets/dataset.csv",
        "project_id": os.environ["GCP_PROJECT_ID"],
        "region": os.environ["GCP_REGION"],
        "display_model_name": "simple-model",            
    },
)

In [51]:
SERVICE_ACCOUNT = os.environ["GCP_SERVICE_ACCOUNT"]
job.submit(service_account = SERVICE_ACCOUNT)

In [27]:
!gcloud storage ls "gs://my-model-training/pipeline_root/kfp/344969539300/pipeline-20260827124514/package-model_-8304349280047464448/"

gs://my-model-training/pipeline_root/kfp/344969539300/pipeline-20260827124514/package-model_-8304349280047464448/

gs://my-model-training/pipeline_root/kfp/344969539300/pipeline-20260827124514/package-model_-8304349280047464448/:
gs://my-model-training/pipeline_root/kfp/344969539300/pipeline-20260827124514/package-model_-8304349280047464448/
gs://my-model-training/pipeline_root/kfp/344969539300/pipeline-20260827124514/package-model_-8304349280047464448/executor_output.json
gs://my-model-training/pipeline_root/kfp/344969539300/pipeline-20260827124514/package-model_-8304349280047464448/model_mar.mar


### Inference

In [52]:
from google.cloud import aiplatform

aiplatform.init(project = PROJECT_ID, location=os.environ["GCP_REGION"])

In [63]:
models = aiplatform.Model.list()

for model in models:
    print("Display Name :", model.display_name)
    print("Model ID     :", model.name)
    print("Resource     :", model.resource_name)
    print("Artifact URI :", model.gca_resource.artifact_uri)
    print("Version ID   :", model.version_id)
    print("-" * 80)

In [64]:
endpoints = aiplatform.Endpoint.list()

for endpoint in endpoints:
    print(f"Nama Display : {endpoint.display_name}")
    print(f"Resource Name: {endpoint.resource_name}")
    print(f"Deployed Models: {len(endpoint.list_models())}")
    print("-" * 40)

In [60]:
response = endpoint.predict(instances=[
        {
            "body": {
                "data": [
                    [1.0, 2.0, 3.0, 4.0],
                    [1.0, 2.0, 3.0, 9.0]
                ]
            }
        }    
])

In [61]:
response

Prediction(predictions=[[[2.472734928131104], [4.89214038848877]]], deployed_model_id='5046895260689498112', metadata=None, model_version_id='1', model_resource_name='projects/344969539300/locations/us-central1/models/1951756834360524800', explanations=None)

In [62]:
endpoint.undeploy_all()
endpoint.delete()
model.delete()